# Experimento do TCC: relações metamórficas em PT-BR (versão gratuita)

Rode as células **de cima para baixo**, uma de cada vez. O passo a passo está no README do repositório.

Antes de começar: copie a pasta do repositório para **Meu Drive** com o nome `tcc` e a chave `GROQ_API_KEY` precisa estar nos **Secrets** do Colab (ícone de chave na barra da esquerda).

**Sempre que abrir o notebook de novo, rode as células 1, 2 e 3 primeiro.**

### Célula 1: conectar o Google Drive e entrar na pasta do projeto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/tcc
!ls

### Célula 2: instalar as bibliotecas

In [ ]:
!pip install -q -r requirements.txt

### Célula 3: carregar as chaves dos Secrets

Só a `GROQ_API_KEY` é obrigatória. As outras são para os modelos pagos opcionais.

In [ ]:
import os
from google.colab import userdata
for nome in ['GROQ_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'DASHSCOPE_API_KEY']:
    try:
        os.environ[nome] = userdata.get(nome)
        print(f'{nome}: carregada')
    except Exception as erro:
        aviso = 'OBRIGATÓRIA, confira o nome no painel de Secrets' if nome == 'GROQ_API_KEY' else 'opcional, pode ignorar'
        print(f'{nome}: não encontrada ({aviso})')

### Célula 4: testar a conexão com o Groq

In [ ]:
!python 00_testar_conexao.py

### Célula 5: preparar os dados

In [ ]:
!python 01_preparar_dados.py

### Célula 6: gerar os casos subsequentes

Se a cota diária do gerador acabar, rode esta célula de novo no dia seguinte: ela continua de onde parou.

In [ ]:
!python 02_gerar_casos.py

### Célula 7: revisão manual, fora do Colab

Nada para rodar aqui. Revise `dados/revisao_manual.xlsx` no Excel ou no Planilhas Google, salve como .xlsx com o mesmo nome dentro de `tcc/dados` e só depois continue.

### Célula 8: piloto do gpt-oss-120b

In [ ]:
!python 03_executar.py --piloto gpt-oss-120b

### Célula 9: execução completa do gpt-oss-120b

Leva de 2 a 3 dias por causa da cota diária gratuita. Quando a cota acabar, a célula para com um aviso. Rode-a de novo no dia seguinte (depois das células 1, 2 e 3) até aparecer o contador completo.

In [ ]:
!python 03_executar.py gpt-oss-120b

### Célula 10: trocar para GPU (modelos locais)

Nada para rodar aqui. Menu **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4** e salve. A sessão reinicia: rode de novo as células 1, 2 e 3 e depois siga para a 11.

### Célula 11: conferir a GPU e instalar as bibliotecas dos modelos locais

In [ ]:
!nvidia-smi
!pip install -q torch transformers accelerate bitsandbytes

### Célula 12: piloto dos três modelos locais

Cada modelo é baixado na primeira vez (de 8 a 16 GB). Os três rodam um depois do outro.

In [ ]:
!python 03_executar.py --piloto qwen3-4b
!python 03_executar.py --piloto qwen3-8b
!python 03_executar.py --piloto tucano2-3.7b

### Células 13, 14 e 15: execução completa dos modelos locais

Uma por vez. Se a sessão cair ou a cota de GPU acabar, rode as células 1, 2, 3 e 11 e depois a mesma célula de novo: ela continua de onde parou.

### Célula 13: Qwen3-4B

In [ ]:
!python 03_executar.py qwen3-4b

### Célula 14: Qwen3-8B

In [ ]:
!python 03_executar.py qwen3-8b

### Célula 15: Tucano 2

In [ ]:
!python 03_executar.py tucano2-3.7b

### Célula 16: análise

In [ ]:
!python 04_analisar.py
from IPython.display import Image
Image('resultados/grafico_violacoes.png')

### Célula 17: empacotar os resultados

Cria `resultados_tcc.zip` na pasta do projeto no Drive. O pacote não inclui as suas chaves.

In [ ]:
!rm -f resultados_tcc.zip && zip -r -q resultados_tcc.zip dados resultados && ls -lh resultados_tcc.zip

## Análise de sensibilidade à instrução (opcional)

Testa se o viés dos modelos menores para a resposta NÃO vem da instrução original. A instrução alternativa (p2) pergunta se *uma pessoa que lê a Frase 1 concluiria que a Frase 2 também é verdadeira*, critério próximo ao da anotação do ASSIN 2. As respostas vão para `resultados/respostas_p2.jsonl`; os resultados originais **não são alterados**.

Precisa da GPU (Célula 10). Se o ambiente foi reiniciado, rode antes as Células 1, 2 e 11.

### Célula 18: teste rápido (só os 100 casos fonte)

Cerca de 10 a 15 minutos por modelo. Ao final, a tabela mostra a coluna `respostas_SIM_%` das duas instruções. **Se ela subir bastante na p2** (por exemplo, de 6–11% para 30% ou mais), vale fazer a execução completa (Células 19 a 21). Se ficar parecida, o viés é do próprio modelo: anote o resultado e pule para a Célula 22.

In [ ]:
!python 03_executar.py --prompt p2 --so-fonte qwen3-4b
!python 03_executar.py --prompt p2 --so-fonte qwen3-8b
!python 03_executar.py --prompt p2 --so-fonte tucano2-3.7b
!python 05_comparar_instrucoes.py

### Células 19, 20 e 21: execução completa com a instrução p2

Reaproveita os casos fonte já rodados. Cada célula leva aproximadamente o mesmo tempo da execução original do modelo. Se cair no meio, rode a mesma célula de novo: ela continua de onde parou.

### Célula 19: Qwen3-4B com p2

In [ ]:
!python 03_executar.py --prompt p2 qwen3-4b

### Célula 20: Qwen3-8B com p2

In [ ]:
!python 03_executar.py --prompt p2 qwen3-8b

### Célula 21: Tucano 2 com p2

In [ ]:
!python 03_executar.py --prompt p2 tucano2-3.7b

### Célula 22: análise da instrução p2 e comparação

Gera as tabelas da p2 em `resultados/p2/` e a comparação em `resultados/tabela_sensibilidade_instrucao.csv`. Depois, rode de novo a Célula 17 para empacotar tudo.

In [ ]:
import os
if os.path.exists('resultados/respostas_p2.jsonl'):
    !python 04_analisar.py --prompt p2
!python 05_comparar_instrucoes.py

### Opcional: modelos pagos (extensão futura)

Cadastre as chaves pagas nos Secrets, rode as células 1, 2 e 3 e depois, por exemplo: `!python 03_executar.py gpt-5.6-terra`. Em seguida, rode a célula 16 de novo.